# ChaoSSM - Phase 17: why the edge wins on Mackey-Glass but not elsewhere

**The puzzle.** Phase 16 found the edge arm beating the ordered arm on Mackey-Glass by +0.061
(CI [+0.047,+0.075], 5/5 seeds) - the opposite sign from the n=20 logistic control (+0.011 favouring
*ordered*) and from all six systems of Phase 13d. Criticality helps on one system and not the others.

**The hypothesis.** The edge reservoir has more linear short-term memory than the ordered one
(Phase 14: MC 6.05 vs 5.55). That extra memory is worthless on a task whose target depends only on the
immediately preceding value, and valuable on a task whose target depends on the distant past.
Mackey-Glass is a *delay*-differential system with tau = 17, so x(t+1) depends on both x(t) and
x(t-17); the logistic map is one-step Markov. If memory demand is the moderator, the edge should win
exactly where the task reaches back further than the ordered reservoir can hold.

**What this notebook measures.** Two things, neither requiring any training:

1. **Memory capacity resolved by delay, MC(k)** - not the usual summed scalar but the whole curve, for
   ordered / edge / deep / linear. The summed MC hides where the memory lives. If the hypothesis holds,
   ordered and edge should be comparable at short delays and diverge at long ones.
2. **The memory demand of each series** - how far back the target actually depends on the input, by
   lagged mutual information and single-lag predictability, for Mackey-Glass, the logistic map and the
   non-chaotic control. This is a property of the data, not the reservoir.

The prediction is specific and falsifiable: MG's dependence should extend to ~tau = 17, the logistic
map's should die after one step, and the edge's MC advantage should sit at precisely the delays MG needs.
If the MC curves are identical at long delay, or MG shows no long-lag dependence, the hypothesis is wrong
and we report the Phase 16 result as an unexplained system-dependence.

**Cost.** No training at all - frozen reservoirs, linear algebra only. A few minutes.

## Setup

In [ ]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field
from itertools import product
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))
def setup_results_dir(project="ChaoSSM",phase="phase17_memory_demand"):
    """Resolve the output directory, preferring Google Drive so results survive the runtime.

    Colab's drive.mount() can fail (e.g. ValueError when /content/drive already holds files, or
    when the mount is refused). Earlier versions silently fell back to the VM's local disk, where
    results are lost when the runtime recycles. We now retry the mount once with force_remount and,
    if Drive is still unavailable, fall back to local disk with a loud warning."""
    base=None
    try:
        from google.colab import drive
        try:
            drive.mount("/content/drive",force_remount=False)
        except Exception as e1:
            print(f"[warn] drive.mount failed ({type(e1).__name__}: {e1}); retrying with force_remount=True")
            drive.mount("/content/drive",force_remount=True)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"/content/{project}_results/{phase}") if os.path.isdir("/content") \
             else os.path.abspath(f"./{project}_results/{phase}")
        print("="*78)
        print(f"[WARNING] Google Drive is NOT mounted ({type(e).__name__}).")
        print(f"[WARNING] Results will be written to {base}")
        print("[WARNING] That is the runtime's local disk -- the files are LOST when the runtime")
        print("[WARNING] recycles. Download them, or copy them to Drive, as soon as the run ends.")
        print("="*78)
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
# NOTE: this notebook writes ONLY to its own results/phase17_memory_demand/ folder.
# It creates new phase14_*.csv / .png files and does NOT touch or overwrite any existing
# Phase 10/13/14 (or other phase) results. Nothing prior is modified.
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

## Configuration

In [ ]:
SMOKE_TEST=False

@dataclass
class Config:
    state_dim:int=64; d_model:int=64
    seeds:int=5
    seq_len:int=64; fc_bins:int=32
    r_cap_map:float=4.0; gamma_init:float=1.0
    lambda_target:dict=field(default_factory=lambda: dict(ordered=-0.30, edge=-0.02, deep=0.30))
    lambda_tol:float=0.05; calib_steps:int=200; calib_warm:int=60
    chaos_regimes:tuple=("ordered","edge","deep")
    # MC probe
    mc_T:int=4000; mc_washout:int=300; mc_max_delay:int=40
    series:tuple=("mackey_glass","logistic","nonchaotic_control")

CFG=Config()
if SMOKE_TEST:
    CFG.seeds=2; CFG.calib_steps=30; CFG.calib_warm=10; CFG.mc_T=600; CFG.mc_max_delay=12
    print("*** SMOKE TEST ***")
print("regimes:",CFG.chaos_regimes,"| max delay:",CFG.mc_max_delay,"| seeds:",CFG.seeds)

## Cells and calibration (verbatim from Phase 14/15/16)

In [ ]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

class ChaosCell(nn.Module):
    def __init__(self,cfg,regime):
        super().__init__(); self.N=cfg.state_dim; self.regime=regime
        self.r_cap=float(cfg.r_cap_map)
        self.w_r=nn.Parameter(torch.zeros(self.N))
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(cfg.gamma_init)))))
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad(): self.w_gamma.copy_(torch.tensor(float(math.log(math.expm1(max(val,1e-3))))))
    def phi(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)
    def step(self,h,x): return triangle_fold(self.phi(h)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze_recurrence(self):
        for p_ in [self.w_r,self.W_mix,self.w_gamma]: p_.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class LinearCell(nn.Module):
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.9)
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def step(self,h,x): return torch.tanh(h@self.W_lin.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): self.W_lin.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class GRUCellArm(nn.Module):
    def __init__(self,cfg,**kw): super().__init__(); self.N=cfg.state_dim; self.cell=nn.GRUCell(cfg.d_model,cfg.state_dim)
    def step(self,h,x): return self.cell(x,h)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): pass
    def freeze_input(self): pass

def make_cell(cfg,cond):
    if cond["arm"]=="gru": return GRUCellArm(cfg)
    if cond["arm"]=="linear": return LinearCell(cfg)
    return ChaosCell(cfg,cond["regime"])

In [ ]:
from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0,steps=None,warm=None):
    steps=steps or cfg.calib_steps; warm=warm or cfg.calib_warm; N=cfg.state_dim
    dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z):
            r=cell.get_r(); u=z@cell.W_mix.T+r*z*(1-z); return triangle_fold(cell.get_gamma()*u)
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev); h=torch.clamp(h,0.0,1.0)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate_gamma(cell,cfg,regime,seed=0):
    # lo widened from Phase 14's 0.05 to 0.01: the most ordered target (-0.60) needs a smaller gain
    # than any Phase 14 target did. Returns a calibration-success flag so that points the cell
    # cannot actually reach are reported as such rather than silently mislabeled.
    target=cfg.lambda_target[regime]; lo,hi=0.01,8.0; best=None; best_err=1e9
    for _ in range(18):
        mid=math.sqrt(lo*hi); cell.set_gamma(mid); lam=map_lyapunov(cell,cfg,seed=seed)
        err=abs(lam-target)
        if err<best_err: best_err=err; best=(mid,lam)
        if lam<target: lo=mid
        else: hi=mid
        if err<cfg.lambda_tol: break
    cell.set_gamma(best[0]); return best[0],best[1],bool(best_err<cfg.lambda_tol)


## Source series

Mackey-Glass and the non-chaotic control come verbatim from the pilot; the logistic map is added as the reference one-step-Markov system used in the main experiments.

In [ ]:
def mackey_glass_series(n, tau=17, beta=0.2, gamma=0.1, nexp=10, dt=1.0, seed=0, discard=500):
    rng=np.random.default_rng(seed); hist=int(tau/dt)+1
    x=np.zeros(n+hist+discard); x[:hist]=1.2+0.2*rng.standard_normal(hist)
    for t in range(hist,len(x)):
        xt=x[t-hist]; x[t]=x[t-1]+dt*(beta*xt/(1+xt**nexp)-gamma*x[t-1])
    return x[hist+discard:]

def nonchaotic_series(n, seed=0):
    rng=np.random.default_rng(seed); t=np.arange(n)
    freqs=[0.017,0.041,0.073]; phases=rng.uniform(0,2*np.pi,len(freqs))
    return sum(np.sin(2*np.pi*f*t+p) for f,p in zip(freqs,phases))+0.15*rng.standard_normal(n)

def normalize01(x):
    lo,hi=x.min(),x.max(); return (x-lo)/(hi-lo+1e-9)

_SRC={}
def get_source(name, seed, length=60000):
    if (name,seed) not in _SRC:
        raw=mackey_glass_series(length,seed=seed) if name=="mackey_glass" else nonchaotic_series(length,seed=seed)
        _SRC[(name,seed)]=normalize01(raw)
    return _SRC[(name,seed)]

def make_forecast_real(cfg,batch,device,gen,series_name,src_seed):
    L=cfg.seq_len; src=get_source(series_name,src_seed); n=len(src)
    # NOTE: torch.randint allocates on CPU by default, so passing a CUDA generator raises
    # "Expected a 'cpu' device type for generator but found 'cuda'". Allocate on the
    # generator's own device so this works identically on CPU and GPU runtimes.
    g=np.random.default_rng(int(torch.randint(0,2**31-1,(1,),generator=gen,device=gen.device).item()))
    starts=g.integers(0,n-L-1,size=batch)
    X=torch.tensor(np.stack([src[s:s+L+1] for s in starts]),dtype=torch.float32,device=device)
    inp=X[:,:L]; tgt=torch.clamp((X[:,1:L+1]*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp,tgt

def majority_chance(cfg,name,src_seed):
    src=get_source(name,src_seed); b=np.clip((src[1:]*cfg.fc_bins).astype(int),0,cfg.fc_bins-1)
    cnt=np.bincount(b,minlength=cfg.fc_bins); return float(cnt.max())/float(cnt.sum())

for nm in ("mackey_glass","nonchaotic_control"):
    s=get_source(nm,0,5000)
    print(f"{nm:20s} range=[{s.min():.3f},{s.max():.3f}] std={s.std():.3f} "
          f"majority-class chance={majority_chance(CFG,nm,0):.4f}")

def logistic_series(n, r=3.9, seed=0, discard=500):
    rng=np.random.default_rng(seed); x=np.zeros(n+discard); x[0]=0.1+0.8*rng.random()
    for t in range(1,len(x)): x[t]=r*x[t-1]*(1-x[t-1])
    return x[discard:]

_SRC_ALL={}
def source(name, seed=0, length=40000):
    if (name,seed) not in _SRC_ALL:
        if name=="mackey_glass": raw=mackey_glass_series(length,seed=seed)
        elif name=="logistic":   raw=logistic_series(length,seed=seed)
        else:                    raw=nonchaotic_series(length,seed=seed)
        _SRC_ALL[(name,seed)]=normalize01(raw)
    return _SRC_ALL[(name,seed)]
for nm in CFG.series:
    s=source(nm,0,5000); print(f"{nm:20s} std={s.std():.3f}")

## Part 1 - how far back does each series actually reach?

For each series we compute, at every lag k, the mutual information between x(t+1) and x(t-k), and the
single-lag predictability of x(t+1) from x(t-k) alone (squared correlation on binned values). Both are
properties of the data and involve no reservoir. A one-step-Markov system should show a sharp drop after
k = 0; a delay system should show a second peak near its delay.

In [ ]:
def lagged_mi(x, k, bins=24):
    a=x[:-(k+1)] if k>0 else x[:-1]
    b=x[k+1:]
    n=min(len(a),len(b)); a,b=a[:n],b[:n]
    ai=np.clip((a*bins).astype(int),0,bins-1); bi=np.clip((b*bins).astype(int),0,bins-1)
    joint=np.zeros((bins,bins))
    np.add.at(joint,(ai,bi),1.0); joint/=joint.sum()
    pa=joint.sum(1,keepdims=True); pb=joint.sum(0,keepdims=True)
    nz=joint>0
    return float((joint[nz]*np.log(joint[nz]/(pa@pb)[nz])).sum())

def lagged_r2(x,k):
    a=x[:-(k+1)] if k>0 else x[:-1]; b=x[k+1:]
    n=min(len(a),len(b)); a,b=a[:n],b[:n]
    va=a-a.mean(); vb=b-b.mean()
    return float((va@vb)**2/((va@va)*(vb@vb)+1e-12))

DEM=[]
for nm in CFG.series:
    s=source(nm,0,20000)
    for k in range(0,CFG.mc_max_delay+1):
        DEM.append(dict(series=nm,lag=k,mutual_info=round(lagged_mi(s,k),4),r2=round(lagged_r2(s,k),4)))
DEMAND=pd.DataFrame(DEM); savetable(DEMAND,"phase17_series_memory_demand.csv")
print("lagged mutual information (nats), selected lags:")
piv=DEMAND.pivot(index="lag",columns="series",values="mutual_info")
print(piv.loc[[k for k in [0,1,2,4,8,12,16,17,18,20,24,32] if k<=CFG.mc_max_delay]].to_string())
for nm in CFG.series:
    d=DEMAND[DEMAND.series==nm]
    far=d[d.lag>=2]
    print(f"{nm:20s} MI(lag0)={d[d.lag==0].mutual_info.iloc[0]:.3f} "
          f"max MI beyond lag1={far.mutual_info.max():.3f} at lag {int(far.loc[far.mutual_info.idxmax()].lag)}")

## Part 2 - memory capacity resolved by delay

The usual MC is a single number: the sum over delays of how well a linear readout recovers u(t-k) from
the frozen reservoir state. Here we keep the whole curve. The reservoir is driven with i.i.d. uniform
input, the canonical probe, so this is a property of the regime alone and not of any task.

In [ ]:
@torch.no_grad()
def mc_profile(cell, cfg, encode_fn, seed=0, drive=None):
    """Per-delay memory capacity r^2(k). drive=None uses i.i.d. uniform input (canonical probe);
    passing a series instead measures retention under that series' own statistics."""
    dev=next(p.device for p in cell.parameters()); T=cfg.mc_T
    g=torch.Generator(device=dev); g.manual_seed(seed)
    if drive is None:
        u=torch.rand(T,generator=g,device=dev)
    else:
        st=int(torch.randint(0,len(drive)-T-1,(1,),generator=g,device=dev).item())
        u=torch.tensor(drive[st:st+T],dtype=torch.float32,device=dev)
    e=encode_fn(u.view(1,-1)).squeeze(0)
    h=cell.init_state(1,dev,torch.float32).squeeze(0); H=[]
    for t in range(T):
        h=triangle_fold(cell.phi(h)+e[t]@cell.B.T) if hasattr(cell,"W_mix") \
          else torch.tanh(h@cell.W_lin.T+e[t]@cell.B.T)
        H.append(h)
    H=torch.stack(H); Hd=H[cfg.mc_washout:]; um=u[cfg.mc_washout:]
    X=torch.cat([Hd,torch.ones(Hd.shape[0],1,device=dev)],1)
    per=[]
    for k in range(1,cfg.mc_max_delay+1):
        if k>=Hd.shape[0]-2: break
        Xk=X[k:]; yk=um[:-k]
        A=Xk.T@Xk+1e-3*torch.eye(Xk.shape[1],device=dev); b=Xk.T@yk
        w=torch.linalg.lstsq(A,b.unsqueeze(1)).solution.squeeze(1)
        pr=Xk@w; vp=pr-pr.mean(); vy=yk-yk.mean()
        per.append(float(torch.clamp((vp@vy)**2/((vp@vp)*(vy@vy)+1e-12),0,1)))
    return per

rows=[]
for reg in list(CFG.chaos_regimes)+["linear"]:
    for sd in range(CFG.seeds):
        torch.manual_seed(sd); np.random.seed(sd)
        if reg=="linear":
            cell=LinearCell(CFG).to(DEVICE); lam=float("nan")
        else:
            cell=ChaosCell(CFG,reg).to(DEVICE); _,lam,_=calibrate_gamma(cell,CFG,reg,seed=sd)
        enc=nn.Linear(1,CFG.d_model).to(DEVICE)          # frozen random encoder, as in Phase 16
        for p_ in enc.parameters(): p_.requires_grad_(False)
        # mc_profile passes a (1,T) tensor; the encoder maps a scalar per step, so add the
        # feature axis exactly as ReservoirLM.forward does (this wrapper is what Phase 14
        # got for free by passing the model's own .encode method)
        encode_fn=lambda x, _e=enc: _e(x.unsqueeze(-1))
        per=mc_profile(cell,CFG,encode_fn,seed=sd)
        for k,r2 in enumerate(per,start=1):
            rows.append(dict(regime=reg,seed=sd,lag=k,r2=round(r2,5),achieved_lambda=lam))
        print(f"  {reg:8s} s={sd} total MC={sum(per):.3f} lam={lam:+.3f}")
MC=pd.DataFrame(rows); savetable(MC,"phase17_mc_profile.csv")

print("\nMC summary by regime:")
agg=MC.groupby(["regime","seed"]).r2.sum().groupby("regime").agg(["mean","std"])
short=MC[MC.lag<=3].groupby(["regime","seed"]).r2.sum().groupby("regime").mean()
long=MC[MC.lag>=10].groupby(["regime","seed"]).r2.sum().groupby("regime").mean()
S=pd.DataFrame(dict(total_MC=agg["mean"].round(3),sd=agg["std"].round(3),
                    MC_lag1_3=short.round(3),MC_lag10plus=long.round(3)))
print(S.to_string()); savetable(S.reset_index(),"phase17_mc_summary.csv")
print("\nkey comparison -- edge minus ordered:")
for col,name in [("total_MC","total"),("MC_lag1_3","short delays (1-3)"),("MC_lag10plus","long delays (>=10)")]:
    print(f"  {name:22s} {S.loc['edge',col]-S.loc['ordered',col]:+.3f}")

## Part 3 - figure and verdict

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4.6))
cols={"ordered":"#4C72B0","edge":"#2E7D32","deep":"#B71C1C","linear":"#9E9E9E"}
ax=axes[0]
for reg in list(CFG.chaos_regimes)+["linear"]:
    d=MC[MC.regime==reg].groupby("lag").r2.agg(["mean","std"])
    ax.plot(d.index,d["mean"],marker="o",ms=4,lw=1.8,color=cols[reg],label=reg)
    ax.fill_between(d.index,d["mean"]-d["std"],d["mean"]+d["std"],color=cols[reg],alpha=.15)
ax.set_xlabel("delay $k$ (steps)"); ax.set_ylabel("$r^2(k)$")
ax.set_title("Memory capacity resolved by delay"); ax.legend(fontsize=9); ax.grid(alpha=.25)
ax=axes[1]
sc={"mackey_glass":"#8456B0","logistic":"#E8900C","nonchaotic_control":"#555555"}
for nm in CFG.series:
    d=DEMAND[DEMAND.series==nm]
    ax.plot(d.lag,d.mutual_info,marker="s",ms=4,lw=1.8,color=sc[nm],label=nm.replace("_"," "))
ax.axvline(17,ls=":",color="#8456B0",lw=1.4)
ax.annotate("Mackey--Glass $\\tau=17$",xy=(17,ax.get_ylim()[1]*0.85),fontsize=9,color="#8456B0",ha="center")
ax.set_xlabel("lag $k$ (steps)"); ax.set_ylabel("mutual information with $x(t{+}1)$ (nats)")
ax.set_title("How far back each target reaches"); ax.legend(fontsize=9); ax.grid(alpha=.25)
fig.suptitle("Does the edge win where the task needs memory?",fontsize=13)
fig.tight_layout(); savefig(fig,"phase17_memory_demand.png"); plt.close(fig)

d_long=S.loc["edge","MC_lag10plus"]-S.loc["ordered","MC_lag10plus"]
d_short=S.loc["edge","MC_lag1_3"]-S.loc["ordered","MC_lag1_3"]
# Compare LONG-lag dependence, matching the MC_lag10plus bucket. Using "max MI beyond lag 1"
# would be wrong here: for a deterministic map, x(t+1) stays informative about x(t-2) simply
# because the map is invertible over short horizons, so both series look similar at lag 2.
# The hypothesis is about the distant past, so that is what we measure.
mg=DEMAND[(DEMAND.series=="mackey_glass")&(DEMAND.lag>=10)].mutual_info.mean()
lg=DEMAND[(DEMAND.series=="logistic")&(DEMAND.lag>=10)].mutual_info.mean()
mg_peak=DEMAND[(DEMAND.series=="mackey_glass")&(DEMAND.lag>=10)]
mg_peak=int(mg_peak.loc[mg_peak.mutual_info.idxmax()].lag) if len(mg_peak) else -1
print("="*76)
if d_long>d_short and mg>lg:
    print("SUPPORTS the hypothesis: the edge's memory advantage is concentrated at long delays,")
    print("and Mackey-Glass retains dependence at long lags where the logistic map does not. The edge wins on MG because")
    print("MG needs memory the ordered reservoir does not hold.")
elif d_long<=d_short:
    print("AGAINST the hypothesis: the edge's memory advantage is NOT concentrated at long delays.")
    print("Memory demand does not explain the Phase 16 result; report it as unexplained system-dependence.")
else:
    print("MIXED: inspect the curves. The MC asymmetry and the series' reach do not line up cleanly.")
print(f"  edge-ordered MC at long delays (>=10): {d_long:+.3f}  | at short delays (1-3): {d_short:+.3f}")
print(f"  mean MI at lags >=10: Mackey-Glass {mg:.3f} vs logistic {lg:.3f}  "
      f"(MG peak in that range at lag {mg_peak}; tau=17)")
print("="*76)

## Rescue results if Drive was not mounted

In [ ]:
import shutil, os
if not RESULTS.startswith("/content/drive"):
    _z=shutil.make_archive("/content/phase17_results","zip",RESULTS)
    print("\nDrive not mounted - zip written to:",_z)
    print("  from google.colab import files"); print(f"  files.download('{_z}')")
else:
    print("\nresults on Drive:",RESULTS)